In [1]:
import shutil
from pathlib import Path

from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_pdfmuse import PdfmuseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

docs = PdfmuseLoader("data/sampled_schemes.pdf").load()
chunks = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=0).split_documents(docs)
print(len(chunks), "chunks from", len(docs), "pages")

VECTOR_STORES_DIR = Path("vector_stores")
shutil.rmtree(VECTOR_STORES_DIR / "qdrant_metadata_db", ignore_errors=True)  # clean slate
VECTOR_STORES_DIR.mkdir(exist_ok=True)

c:\Users\hp\Desktop\genai python\genai_class\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


39 chunks from 9 pages


In [2]:
docs[3].metadata

{'source': 'data/sampled_schemes.pdf',
 'source_kind': 'Pdf',
 'page': 3,
 'total_pages': 9}

In [3]:
chunks[0].metadata

{'source': 'data/sampled_schemes.pdf',
 'source_kind': 'Pdf',
 'page': 0,
 'total_pages': 9}

In [4]:
print(chunks[0].page_content)

CENTRALLY SPONSORED SCHEMES 

NATIONAL FOOD SECURITY MISSION 

The scheme is being implemented from 2007­08 as a Central Sector Scheme with an  objective to enhance production & productivity of Rice & Pulse crops in selected districts  of  the  state  in  Mission  Mode.  Besides,  this  embodies  Sub­Schemes  for  enhancing  production & productivity of Coarse Cereals under NFSM­Coarse Cereals, and Cotton,  Jute  and  Sugarcane  under  NFSM­Commercial  crops,  NFSM­Nutricereals  and  NFSM­ Oilseeds. Besides, Targeting Rice Fallow Areas is being implemented under NFSM for  increasing coverage under pulses and oilseeds crops.


In [5]:
from collections import Counter

PAGE_METADATA = {
    0: ("Crop Production", "National Food Security Mission (NFSM)"),
    1: ("Soil Health & Sustainable Agriculture", "National Mission for Sustainable Agriculture (NMSA)"),
    2: ("Soil Health & Sustainable Agriculture", "National Mission for Sustainable Agriculture (NMSA)"),
    3: ("Extension & Technology", "National Mission on Agricultural Extension and Technology (NMAET)"),
    4: ("Extension & Technology", "National Mission on Agricultural Extension and Technology (NMAET)"),
    5: ("Crop Insurance", "Pradhan Mantri Fasal Bima Yojana (PMFBY)"),
    6: ("Crop Insurance", "Pradhan Mantri Fasal Bima Yojana (PMFBY)"),
    7: ("Crop Insurance", "Pradhan Mantri Fasal Bima Yojana (PMFBY)"),
    8: ("Crop Insurance", "Pradhan Mantri Fasal Bima Yojana (PMFBY)"),
}



In [6]:
for chunk in chunks:
    category, scheme_name = PAGE_METADATA[chunk.metadata["page"]]
    chunk.metadata["category"] = category
    chunk.metadata["scheme_name"] = scheme_name



In [7]:
print(chunks[37].metadata)


{'source': 'data/sampled_schemes.pdf', 'source_kind': 'Pdf', 'page': 8, 'total_pages': 9, 'category': 'Crop Insurance', 'scheme_name': 'Pradhan Mantri Fasal Bima Yojana (PMFBY)'}


In [8]:
chunks[0].metadata

{'source': 'data/sampled_schemes.pdf',
 'source_kind': 'Pdf',
 'page': 0,
 'total_pages': 9,
 'category': 'Crop Production',
 'scheme_name': 'National Food Security Mission (NFSM)'}

In [9]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

qdrant_client = QdrantClient(path=str(VECTOR_STORES_DIR / "qdrant_metadata_db"))
vector_size = len(embeddings.embed_query("sample text"))

qdrant_client.create_collection(
    collection_name="schemes_meta",
    vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE),
)

qdrant_store = QdrantVectorStore(client=qdrant_client, collection_name="schemes_meta", embedding=embeddings)
qdrant_store.add_documents(chunks)
print("indexed", qdrant_client.get_collection("schemes_meta").points_count, "points")

indexed 39 points


In [10]:
QUERY_DAMAGE = "How is the government helping improve soil and crop productivity?"

for doc, score in qdrant_store.similarity_search_with_score(QUERY_DAMAGE, k=10):
    print(f"[{score:.4f}] ({doc.metadata['category']}) {doc.page_content[:90].replace(chr(10), ' ')}")

[0.5599] (Crop Production)   Cluster Demonstrations – test beds for demonstrations are being set up on  improved pac
[0.5552] (Crop Insurance) 2.     2.1   2.2   2.3   2.4   2.5        Encouragingfarmerstoadoptinnovativeandmodernagr
[0.5175] (Crop Insurance) States shall adopt technology, such as satellite and UAV remote sensing,  for various appl
[0.5138] (Soil Health & Sustainable Agriculture) Interventions:  Provision  of  subsidy  to  farmers  on  the  selected  improved  farm  ma
[0.5032] (Crop Production) Objectives: Increase food security by stepping up the overall food production and food  st
[0.5028] (Extension & Technology) 2.  SUB­MISSION ON SEED AND PLANTING MATERIAL (SMSP)      Objective:  The  scheme  is  bei
[0.5002] (Soil Health & Sustainable Agriculture)  Soil Health Card   Demon./ Fin. Assistance to Farmers   Farmers’Training   Campaign M
[0.4945] (Soil Health & Sustainable Agriculture) NATIONAL MISSION FOR SUSTAINABLE AGRICULTURE (NMSA)     The scheme is being 

In [11]:
from qdrant_client.models import Filter, FieldCondition, MatchValue, MatchAny, Range

insurance_only = Filter(
    must=[FieldCondition(key="metadata.category", match=MatchValue(value="Crop Insurance"))]
)

for doc, score in qdrant_store.similarity_search_with_score(QUERY_DAMAGE, k=10, filter=insurance_only):
    print(f"[{score:.4f}] ({doc.metadata['category']}) {doc.page_content[:90].replace(chr(10), ' ')}")

[0.5552] (Crop Insurance) 2.     2.1   2.2   2.3   2.4   2.5        Encouragingfarmerstoadoptinnovativeandmodernagr
[0.5175] (Crop Insurance) States shall adopt technology, such as satellite and UAV remote sensing,  for various appl
[0.4478] (Crop Insurance) In  an  endeavour  to  integrate  Technology  in  implementation  and  execution of the Sc
[0.4258] (Crop Insurance) Insurance  Companies  shall  compulsorily  use  technology/mobile  applications  for  moni
[0.3894] (Crop Insurance) All data pertaining to crop­wise, area­wise historical yield data, weather  data, sown are
[0.3762] (Crop Insurance) All  farmers  including  sharecroppers  and  tenant  farmers  growing  the  notified crops
[0.3631] (Crop Insurance) Implementing States and Insurance Companies during each crop season  are required to digit
[0.3628] (Crop Insurance) Voluntary Component    The Scheme is optional for non­loanee farmers.    The  insurance 
[0.3499] (Crop Insurance) 3.1.1.2   3.1.1.3   3.1.2   3.1.3   3.

In [12]:

PAGE_METADATA = {
    0: ("Crop Production", "National Food Security Mission (NFSM)"),
    1: ("Soil Health & Sustainable Agriculture", "National Mission for Sustainable Agriculture (NMSA)"),
    2: ("Soil Health & Sustainable Agriculture", "National Mission for Sustainable Agriculture (NMSA)"),
    3: ("Extension & Technology", "National Mission on Agricultural Extension and Technology (NMAET)"),
    4: ("Extension & Technology", "National Mission on Agricultural Extension and Technology (NMAET)"),
    5: ("Crop Insurance", "Pradhan Mantri Fasal Bima Yojana (PMFBY)"),
    6: ("Crop Insurance", "Pradhan Mantri Fasal Bima Yojana (PMFBY)"),
    7: ("Crop Insurance", "Pradhan Mantri Fasal Bima Yojana (PMFBY)"),
    8: ("Crop Insurance", "Pradhan Mantri Fasal Bima Yojana (PMFBY)"),
}


In [13]:
QUERY_SOIL = "How is the government helping improve soil and crop productivity?"

production_or_soil = Filter(
    must=[FieldCondition(
        key="metadata.category",
        match=MatchAny(any=["Crop Production", "Soil Health & Sustainable Agriculture"]),
    )]
)

for doc, score in qdrant_store.similarity_search_with_score(QUERY_SOIL, k=10, filter=production_or_soil):
    print(f"[{score:.4f}] ({doc.metadata['category']}) {doc.page_content[:90].replace(chr(10), ' ')}")

[0.5599] (Crop Production)   Cluster Demonstrations – test beds for demonstrations are being set up on  improved pac
[0.5138] (Soil Health & Sustainable Agriculture) Interventions:  Provision  of  subsidy  to  farmers  on  the  selected  improved  farm  ma
[0.5032] (Crop Production) Objectives: Increase food security by stepping up the overall food production and food  st
[0.5002] (Soil Health & Sustainable Agriculture)  Soil Health Card   Demon./ Fin. Assistance to Farmers   Farmers’Training   Campaign M
[0.4945] (Soil Health & Sustainable Agriculture) NATIONAL MISSION FOR SUSTAINABLE AGRICULTURE (NMSA)     The scheme is being implemented wi
[0.4760] (Crop Production) CENTRALLY SPONSORED SCHEMES   NATIONAL FOOD SECURITY MISSION   The scheme is being impleme
[0.4641] (Soil Health & Sustainable Agriculture) Objective:  The  aim  of  the  Mission  is  to  restructure  and  strengthen  agricultural
[0.4343] (Soil Health & Sustainable Agriculture) o  PGS Certification and Quality Cont

In [14]:
QUERY_GENERIC = "How does the scheme work?"

later_pages_only = Filter(
    must=[FieldCondition(key="metadata.page", range=Range(gte=5))]
)

for doc, score in qdrant_store.similarity_search_with_score(QUERY_GENERIC, k=4, filter=later_pages_only):
    print(f"[{score:.4f}] page={doc.metadata['page']} ({doc.metadata['category']})")

[0.4057] page=8 (Crop Insurance)
[0.3877] page=6 (Crop Insurance)
[0.3650] page=5 (Crop Insurance)
[0.3462] page=7 (Crop Insurance)


In [15]:
QUERY_TECH = "How is technology used to support farmers?"

extension_excluding_page3 = Filter(
    must=[FieldCondition(key="metadata.category", match=MatchValue(value="Extension & Technology"))],
    must_not=[FieldCondition(key="metadata.page", match=MatchValue(value=3))],
)

for doc, score in qdrant_store.similarity_search_with_score(QUERY_TECH, k=4, filter=extension_excluding_page3):
    print(f"[{score:.4f}] page={doc.metadata['page']} ({doc.metadata['category']})")

[0.4806] page=4 (Extension & Technology)
[0.3365] page=4 (Extension & Technology)
[0.3215] page=4 (Extension & Technology)
[0.2478] page=4 (Extension & Technology)


In [16]:
insurance_retriever = qdrant_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3, "filter": insurance_only},
)
insurance_retriever.invoke(QUERY_DAMAGE)

[Document(metadata={'source': 'data/sampled_schemes.pdf', 'source_kind': 'Pdf', 'page': 5, 'total_pages': 9, 'category': 'Crop Insurance', 'scheme_name': 'Pradhan Mantri Fasal Bima Yojana (PMFBY)', '_id': 'de0a3693c31e4d679c6e3b0cdafa455f', '_collection_name': 'schemes_meta'}, page_content='2.  \xa0\n\n2.1 \n\n2.2 \n\n2.3 \n\n2.4 \n\n2.5 \n\n\xa0\n\n\uf0b7  Encouragingfarmerstoadoptinnovativeandmodernagriculturalpractices\xa0 \uf0b7  Ensuringflowofcredittotheagriculturesectorwhichwillcontributetofoodse curity,\xa0crop\xa0diversification\xa0and\xa0enhancing\xa0growth\xa0and\xa0competitiveness\xa0of\xa0 agriculture\xa0sector\xa0besides\xa0protecting\xa0farmers\xa0from\xa0production\xa0risks.\xa0\n\nAdoption\xa0of\xa0Technology\xa0for\xa0Scheme\xa0Administration:'),
 Document(metadata={'source': 'data/sampled_schemes.pdf', 'source_kind': 'Pdf', 'page': 7, 'total_pages': 9, 'category': 'Crop Insurance', 'scheme_name': 'Pradhan Mantri Fasal Bima Yojana (PMFBY)', '_id': 'e5e2b943dfcb45df8d1e

In [17]:
from typing import Literal, Optional

from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain.messages import SystemMessage, HumanMessage

llm = ChatOpenAI(model="gpt-5.4-nano", temperature=0)


class SchemeFilter(BaseModel):
    """Structured filter extracted from a natural-language question about the schemes document."""

    category: Optional[Literal[
        "Crop Production",
        "Soil Health & Sustainable Agriculture",
        "Extension & Technology",
        "Crop Insurance",
    ]] = Field(
        default=None,
        description="Set only if the question clearly refers to one specific scheme category. "
        "Leave null for general questions or ones spanning multiple categories.",
    )
    min_page: Optional[int] = Field(
        default=None, description="Lower bound on page number (0-8), only if the question refers to a specific part of the document."
    )
    max_page: Optional[int] = Field(
        default=None, description="Upper bound on page number (0-8), only if applicable."
    )


filter_llm = llm.with_structured_output(SchemeFilter)

FILTER_SYSTEM_PROMPT = """You turn a user's question about a government schemes PDF into a structured filter.
The document has 9 pages (0-8) split into four categories:
- Crop Production (page 0): National Food Security Mission (NFSM)
- Soil Health & Sustainable Agriculture (pages 1-2): National Mission for Sustainable Agriculture (NMSA)
- Extension & Technology (pages 3-4): National Mission on Agricultural Extension and Technology (NMAET)
- Crop Insurance (pages 5-8): Pradhan Mantri Fasal Bima Yojana (PMFBY)

Only set a field when the question clearly implies that constraint. Leave fields null for general questions."""

In [18]:
def build_filter(sf: SchemeFilter) -> Filter | None:
    conditions = []

    if sf.category:
        conditions.append(
            FieldCondition(
                key="metadata.category",
                match=MatchValue(value=sf.category)
            )
        )

    if sf.min_page is not None or sf.max_page is not None:
        conditions.append(
            FieldCondition(
                key="metadata.page",
                range=Range(gte=sf.min_page, lte=sf.max_page)
            )
        )

    return Filter(must=conditions) if conditions else None

In [19]:
query1 = "How is the government helping improve soil and crop productivity?"
query2 = "What are the benefits of crop insurance?"


In [20]:

# Extract filter from the query
extracted = filter_llm.invoke([
    SystemMessage(FILTER_SYSTEM_PROMPT),
    HumanMessage(query1)
])

# Build Qdrant filter
qdrant_filter = build_filter(extracted)



In [21]:
extracted

SchemeFilter(category='Soil Health & Sustainable Agriculture', min_page=1, max_page=2)

In [22]:
extracted.category, extracted.min_page, extracted.max_page

('Soil Health & Sustainable Agriculture', 1, 2)

In [23]:
query = "What are the benefits of crop insurance?"
print(f"query: {query!r}")
print(f"extracted filter: {extracted}")

# Search Qdrant
results = qdrant_store.similarity_search_with_score(
    query,
    k=4,
    filter=qdrant_filter
)

for doc, score in results:
    print(
        f"[{score:.4f}] "
        f"page={doc.metadata['page']} "
        f"({doc.metadata['category']}) "
        f"{doc.page_content[:70].replace(chr(10), ' ')}"
    )

query: 'What are the benefits of crop insurance?'
extracted filter: category='Soil Health & Sustainable Agriculture' min_page=1 max_page=2
[0.3467] page=2 (Soil Health & Sustainable Agriculture) Interventions:  Provision  of  subsidy  to  farmers  on  the  selected
[0.3083] page=1 (Soil Health & Sustainable Agriculture)  Soil Health Card   Demon./ Fin. Assistance to Farmers   Farmers’Tr
[0.2631] page=2 (Soil Health & Sustainable Agriculture) Objective:  The  aim  of  the  Mission  is  to  restructure  and  stre
[0.2547] page=1 (Soil Health & Sustainable Agriculture) NATIONAL MISSION FOR SUSTAINABLE AGRICULTURE (NMSA)     The scheme is 


In [24]:
llm_filtered_search = lambda q: qdrant_store.similarity_search_with_score(
    q,
    k=4,
    filter=build_filter(filter_llm.invoke([
        SystemMessage(FILTER_SYSTEM_PROMPT),
        HumanMessage(q)
    ]))
)
llm_filtered_search("What happens if a farmer's crop is damaged?")
llm_filtered_search("What is covered under NMAET?")
llm_filtered_search("How does the scheme work?")

[(Document(metadata={'source': 'data/sampled_schemes.pdf', 'source_kind': 'Pdf', 'page': 8, 'total_pages': 9, 'category': 'Crop Insurance', 'scheme_name': 'Pradhan Mantri Fasal Bima Yojana (PMFBY)', '_id': '33649f54f87340e08eca346eba8c8a70', '_collection_name': 'schemes_meta'}, page_content='Voluntary\xa0Component\xa0 \uf0b7  The\xa0Scheme\xa0is\xa0optional\xa0for\xa0non\xadloanee\xa0farmers.\xa0 \uf0b7  The\xa0 insurance\xa0 coverage\xa0 will\xa0 strictly\xa0 be\xa0 equivalent\xa0 to\xa0 sum\xa0 insured\xa0 /hectare,\xa0as\xa0defined\xa0in\xa0the\xa0Govt.\xa0notification\xa0or/and\xa0on\xa0National\xa0Crop\xa0 Insurance\xa0Portal\xa0multiplied\xa0by\xa0sown\xa0area\xa0for\xa0notified\xa0crop.\xa0 Special\xa0efforts\xa0shall\xa0be\xa0made\xa0to\xa0ensure\xa0maximum\xa0coverage\xa0of\xa0SC/\xa0ST/\xa0 Women\xa0farmers\xa0under\xa0the\xa0Scheme.\xa0Further\xa0Panchayat\xa0Raj\xa0Institutions\xa0 (PRIs)may\xa0be\xa0involved\xa0in\xa0extension\xa0and\xa0awareness\xa0creation amongst\xa0 fa

In [25]:
query = "What happens if a farmer's crop is damaged?"

# Extract filter from the query
extracted = filter_llm.invoke([
    SystemMessage(FILTER_SYSTEM_PROMPT),
    HumanMessage(query)
])

# Build Qdrant filter
qdrant_filter = build_filter(extracted)

print(f"query: {query!r}")
print(f"extracted filter: {extracted}")

# Search Qdrant
results = qdrant_store.similarity_search_with_score(
    query,
    k=4,
    filter=qdrant_filter
)

for doc, score in results:
    print(
        f"[{score:.4f}] "
        f"page={doc.metadata['page']} "
        f"({doc.metadata['category']}) "
        f"{doc.page_content[:70].replace(chr(10), ' ')}"
    )

query: "What happens if a farmer's crop is damaged?"
extracted filter: category='Crop Insurance' min_page=5 max_page=8
[0.4879] page=7 (Crop Insurance) All  farmers  including  sharecroppers  and  tenant  farmers  growing 
[0.4531] page=8 (Crop Insurance) Merely,  sanctioning  of  crop  loan  against  other  collateral  secu
[0.4478] page=8 (Crop Insurance) 3.1.1.2   3.1.1.3   3.1.2   3.1.3   3.1.4      (SAO) loans from Financ
[0.4314] page=8 (Crop Insurance) Voluntary Component    The Scheme is optional for non­loanee farmers.
